# QUANTUM LAB — Espín total de dos electrones (notación de Dirac)

Dada una función de onda de dos electrones escrita con **espín-orbitales hidrogenoides**, el cuaderno calcula los autovalores de **S²** (momento angular de espín total) y de **S_z**. Si la función **no** es autofunción, muestra la función resultante (S²Ψ, S_zΨ), sus componentes de singlete y triplete, y las probabilidades de cada resultado de medida.

**Criterios del curso que cumple este cuaderno**

- *Autovalor del momento angular de espín total y de su componente z* para cualquier función de dos electrones en espín-orbitales hidrogenoides.
- *Si no es autofunción, dar la función resultante.* Se muestran S²Ψ y S_zΨ, y la descomposición en singlete (s = 0) y triplete (s = 1).
- *Entrada en notación de Dirac (bra-ket).*

**Cómo se escribe Ψ**

- Cada ket es `|orbital,espín ; orbital,espín⟩`: primero el electrón 1, después el electrón 2. Espín: `α` / `β` (o `alpha` / `beta`, `up` / `down`).
- Orbitales hidrogenoides: `1s`, `2s`, `2p0`, `2p+1`, `2p-1`, `3d+2`, `3d-1`… (n, l y mₗ; `2pz` equivale a `2p0`; también `2px`, `2py` y los orbitales reales d).
- Coeficientes: `1/√2`, `1/sqrt(2)`, `2^(-1/2)`, `1/2`, `i`, o símbolos como `c`, `d`. Se admite `Ψ = ...` al principio.
- Determinante de Slater: `Slater(1sα, 2sβ)` equivale a (|1sα;2sβ⟩ − |2sβ;1sα⟩)/√2.
- Ejemplos: `(|1s,α;2s,β⟩ - |1s,β;2s,α⟩)/√2`  ·  `|1sα;2sα⟩`  ·  `Slater(1sα, 1sβ)`

**Qué hay en cada celda**

| Celda | Contenido |
|---|---|
| 1 | Formato de resultados (recuadros, columnas alineadas, tablas) |
| 2 | Lectura de la entrada: orbitales, kets y coeficientes |
| 3 | Operadores S_z y S², autovalores, descomposición y simetría |
| 4 | Informe de resultados en recuadros |
| 5 | Pruebas (comparan con una construcción matricial independiente y con casos conocidos) |
| 6 | **Entrada del usuario**: escribe Ψ |
| 7 | Galería de ejemplos |

**Cómo usarlo en Google Colab**

1. Menú **Entorno de ejecución → Ejecutar todo**. No hace falta instalar nada.
2. Cambia Ψ en la celda 6 y vuelve a ejecutar solo esa celda.
3. Si la entrada no se entiende o un orbital no existe (por ejemplo `1p` o `2p+2`), el cuaderno **no se detiene**: explica el motivo.

**Convención.** Se trabaja en unidades de ħ: S² tiene autovalores s(s+1) ħ², con s = 0 (singlete) o s = 1 (triplete), y S_z tiene autovalores m_s ħ. Las funciones de espín α y β y los espín-orbitales distintos se toman ortonormales.

Para ver el código de una celda, haz doble clic sobre su título.


In [1]:
#@title 1) Utilidades comunes: recuadros, columnas alineadas y tablas { display-mode: "form" }
# =====================================================================================
#  Esta celda no contiene física: solo da formato a los resultados (recuadros, columnas
#  alineadas, tablas) con el mismo estilo de los cuadernos de la caja, el oscilador y la
#  esfera. Ejecútala antes que las demás.
# =====================================================================================
import math
import textwrap
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

ANCHO = 80


def titulo_caja(titulo):
    """Recuadro de título."""
    print()
    print("╔" + "═" * (ANCHO - 2) + "╗")
    print("║" + titulo.center(ANCHO - 2) + "║")
    print("╚" + "═" * (ANCHO - 2) + "╝")


def seccion(titulo):
    """Título de sección con líneas arriba y abajo."""
    print()
    print("─" * ANCHO)
    print(titulo)
    print("─" * ANCHO)


def dato(etiqueta, valor, unidad="", ancho_etiqueta=40):
    """Una línea «etiqueta : valor unidad» con los dos puntos alineados."""
    texto = f"{valor} {unidad}".rstrip()
    print(f"   {etiqueta:<{ancho_etiqueta}}: {texto}")


def parrafo(texto, sangria=3):
    for linea in textwrap.wrap(texto, ANCHO - sangria):
        print(" " * sangria + linea)


def vinetas(lineas, sangria=3):
    for linea in lineas:
        partes = textwrap.wrap(linea, ANCHO - sangria - 2)
        for i, parte in enumerate(partes):
            print(" " * sangria + ("• " if i == 0 else "  ") + parte)


def tabla(encabezados, filas, sangria=5, alinear=None):
    """Tabla con bordes de caja; cada columna se ajusta a su contenido. alinear: «c» centrado, «l» izquierda."""
    filas = [[str(c) for c in f] for f in filas]
    anchos = [max(len(str(h)), *(len(f[i]) for f in filas)) + 4 if filas else len(str(h)) + 4
              for i, h in enumerate(encabezados)]
    margen = " " * sangria

    def linea(izq, cruce, der):
        return margen + izq + cruce.join("─" * a for a in anchos) + der

    alinear = alinear or "c" * len(encabezados)

    def celda(c, a, modo):
        return (" " + c).ljust(a) if modo == "l" else c.center(a)

    print(linea("┌", "┬", "┐"))
    print(margen + "│" + "│".join(str(h).center(a) for h, a in zip(encabezados, anchos)) + "│")
    print(linea("├", "┼", "┤"))
    for f in filas:
        print(margen + "│" + "│".join(celda(c, a, m) for c, a, m in zip(f, anchos, alinear)) + "│")
    print(linea("└", "┴", "┘"))


class Diagnostico:
    """
    Reúne los comentarios sobre los datos introducidos:
      ERROR  no se puede calcular con ese dato (se explica el motivo físico);
      AVISO  se calcula, pero el resultado no tiene respaldo físico o el modelo deja de valer;
      NOTA   información útil para interpretar los resultados.
    """

    def __init__(self):
        self.items = []

    def error(self, texto):
        self.items.append(("ERROR", texto))

    def aviso(self, texto):
        self.items.append(("AVISO", texto))

    def nota(self, texto):
        self.items.append(("NOTA", texto))

    @property
    def hay_error(self):
        return any(nivel == "ERROR" for nivel, _ in self.items)

    def imprimir(self, titulo="VALIDEZ FÍSICA DE LOS DATOS"):
        titulo_caja(titulo)
        if not self.items:
            print("   Todos los valores están dentro del rango de validez del modelo.")
            return
        orden = {"ERROR": 0, "AVISO": 1, "NOTA": 2}
        for nivel, texto in sorted(self.items, key=lambda it: orden[it[0]]):
            etiqueta = f"   [{nivel}] "
            for i, parte in enumerate(textwrap.wrap(texto, ANCHO - len(etiqueta))):
                print((etiqueta if i == 0 else " " * len(etiqueta)) + parte)
            print()
        if self.hay_error:
            print("   >> Con estos datos no se puede calcular. Corrige los valores marcados como ERROR.")


def informe_pruebas(resultados, titulo):
    """Imprime una tabla con las pruebas: nombre, resultado (OK / FALLÓ) y detalle."""
    titulo_caja(titulo)
    filas = [(nombre, "OK" if ok else "FALLÓ", detalle) for nombre, ok, detalle in resultados]
    tabla(["Prueba", "Resultado", "Detalle"], filas, sangria=1, alinear="lcl")
    n_ok = sum(1 for _, ok, _ in resultados if ok)
    print(f"\n   {n_ok} de {len(resultados)} pruebas superadas.")
    return n_ok == len(resultados)


In [2]:
#@title 2) Lectura de la entrada: orbitales, kets y coeficientes { display-mode: "form" }
# =====================================================================================
#  Estado de dos electrones:  Ψ = Σ cₖ |φₐ σₐ ; φᵦ σᵦ⟩   (electrón 1 ; electrón 2)
#  Se guarda como diccionario {(orbital1, espín1, orbital2, espín2): coeficiente}.
# =====================================================================================
import re
from dataclasses import dataclass

import numpy as np
import sympy as sp
from sympy.parsing.sympy_parser import (convert_xor, implicit_multiplication_application, parse_expr,
                                        standard_transformations)

TRANSFORMACIONES = standard_transformations + (implicit_multiplication_application, convert_xor)
_KET_RE = re.compile(r"\|([^|<>]+)>")
_SPIN_RE = re.compile(r"(?i)^(.*?)(alpha|beta|up|down|[αβ])$")
_SLATER_RE = re.compile(r"(?i)slater\s*\(\s*([^,;()|<>]+?)\s*,\s*([^,;()|<>]+?)\s*\)")
LETRAS_L = "spdfghik"
_REALES = {1: {"x", "y", "z"}, 2: {"z2", "xz", "yz", "xy", "x2-y2"}}


@dataclass(frozen=True)
class Ket:
    """Ket de dos electrones, cada uno como (orbital, espín)."""
    orbital_1: str
    spin_1: str
    orbital_2: str
    spin_2: str

    @property
    def base(self):
        return (self.orbital_1, self.spin_1, self.orbital_2, self.spin_2)


def normalizar_orbital(texto):
    """
    Valida un orbital hidrogenoide y lo devuelve en forma canónica.
    Admite 1s, 2p, 2p0, 2p+1, 2p-1, 3d-2, 2pz (≡ 2p0), 2px, 2py, 3dxy, ...
    Reglas: n ≥ 1, 0 ≤ l ≤ n−1, |mₗ| ≤ l.
    """
    t = texto.strip().replace("−", "-").replace(" ", "")
    m = re.fullmatch(r"(?i)(\d+)([spdfghik])(.*)", t)
    if not m:
        raise ValueError(f"Orbital no reconocido: «{texto}». Use el formato n + letra + mₗ opcional, p. ej. 1s, 2p0, 3d+1.")
    n, l, resto = int(m.group(1)), LETRAS_L.index(m.group(2).lower()), m.group(3).lower()
    if n < 1:
        raise ValueError(f"Orbital «{texto}»: n debe ser ≥ 1.")
    if l >= n:
        raise ValueError(f"Orbital «{texto}» no existe: con n = {n} solo se permite l ≤ {n - 1}.")
    letra = LETRAS_L[l]
    if resto == "":
        return f"{n}{letra}"
    if re.fullmatch(r"[+-]?\d+", resto):
        ml = int(resto)
        if abs(ml) > l:
            raise ValueError(f"Orbital «{texto}» no existe: para l = {l} debe cumplirse |mₗ| ≤ {l}.")
        return f"{n}{letra}{ml:+d}" if ml != 0 else f"{n}{letra}0"
    if l in _REALES and resto in _REALES[l]:
        return f"{n}{letra}0" if (l == 1 and resto == "z") else f"{n}{letra}{resto}"
    raise ValueError(f"Orbital «{texto}»: la parte «{resto}» no es un mₗ válido para l = {l}.")


def normalizar_espin(valor):
    v = valor.strip().lower()
    if v in {"alpha", "up", "α"}:
        return "alpha"
    if v in {"beta", "down", "β"}:
        return "beta"
    raise ValueError(f"Espín no reconocido: «{valor}». Use α/β o alpha/beta.")


def _leer_electron(texto):
    texto = texto.strip()
    partes = [p for p in re.split(r"[,:\s]+", texto) if p]
    if len(partes) == 2:
        return normalizar_orbital(partes[0]), normalizar_espin(partes[1])
    m = _SPIN_RE.match(texto)
    if m and m.group(1).strip():
        return normalizar_orbital(m.group(1)), normalizar_espin(m.group(2))
    raise ValueError(f"No se pudo interpretar el electrón «{texto}». Use, por ejemplo, «1s,α» o «1sα».")


def leer_ket(contenido):
    """Convierte «orbital,espín;orbital,espín» en un Ket."""
    electrones = [p.strip() for p in contenido.split(";")]
    if len(electrones) != 2:
        electrones = [p.strip() for p in contenido.split(",")]
    if len(electrones) != 2:
        raise ValueError(f"El ket «{contenido}» debe tener exactamente dos electrones, separados por «;».")
    return Ket(*_leer_electron(electrones[0]), *_leer_electron(electrones[1]))


def preparar_texto(texto):
    """Limpia la entrada: «Ψ = », ⟩, √, Slater(...)."""
    t = texto.strip()
    if "=" in t:
        t = t.split("=", 1)[1]
    t = t.replace("⟩", ">").replace("〉", ">").replace("¦", "|").replace("−", "-")
    t = re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = re.sub(r"√\s*\(", "sqrt(", t)
    return _SLATER_RE.sub(lambda m: f"(1/sqrt(2))*(|{m.group(1)};{m.group(2)}> - |{m.group(2)};{m.group(1)}>)", t)


def _coeficiente(texto, nombres):
    local = {"sqrt": sp.sqrt, "I": sp.I, "i": sp.I}
    local.update({nombre: sp.Symbol(nombre) for nombre in nombres})
    return parse_expr(texto, local_dict=local, transformations=TRANSFORMACIONES, evaluate=True)


def leer_estado(texto):
    """Combinación lineal de kets -> {base: coeficiente}. Cada ket se sustituye por un símbolo KET_k rodeado de espacios."""
    texto = preparar_texto(texto)
    coincidencias = list(_KET_RE.finditer(texto))
    if not coincidencias:
        raise ValueError("No se encontró ningún ket |...> en el estado.")
    kets = [leer_ket(m.group(1)) for m in coincidencias]
    simbolos = [f"KET_{k}" for k in range(len(kets))]
    reescrito = texto
    for m, s in reversed(list(zip(coincidencias, simbolos))):
        reescrito = reescrito[:m.start()] + f" {s} " + reescrito[m.end():]
    try:
        expr = sp.expand(sp.nsimplify(_coeficiente(reescrito, simbolos), rational=True))
    except Exception as error:
        raise ValueError(f"No se pudo interpretar la expresión de los coeficientes ({error}).") from None
    estado, reconstruido = {}, sp.Integer(0)
    for s, ket in zip(simbolos, kets):
        c = sp.simplify(expr.coeff(sp.Symbol(s)))
        reconstruido += c * sp.Symbol(s)
        if c != 0:
            estado[ket.base] = sp.simplify(estado.get(ket.base, 0) + c)
    if sp.simplify(expr - reconstruido) != 0:
        raise ValueError("La expresión debe ser lineal en los kets: no se permiten productos de kets ni términos sueltos.")
    return {b: c for b, c in estado.items() if c != 0}


In [3]:
#@title 3) Física del espín: S_z, S², autovalores, descomposición y simetría { display-mode: "form" }
# =====================================================================================
#  Un electrón: s = 1/2;  ŝ² α = ŝ² β = (3/4)ħ²;  ŝ_z α = +ħ/2 α,  ŝ_z β = −ħ/2 β.
#  Dos electrones:  Ŝ = ŝ₁ + ŝ₂ ,   Ŝ_z = ŝ₁z + ŝ₂z ,
#       Ŝ² = ŝ₁² + ŝ₂² + 2 ŝ₁·ŝ₂ = 3/2 + 2 ŝ₁zŝ₂z + ŝ₁₊ŝ₂₋ + ŝ₁₋ŝ₂₊      (en unidades de ħ²)
#  Sobre la base:  αα y ββ → 2 (triplete);  αβ → 1·αβ + βα  (el término ŝ₁₊ŝ₂₋ + ŝ₁₋ŝ₂₊ intercambia los espines).
#  Autovalores de Ŝ²: s(s+1)ħ² con s = 0 (singlete, 0) o s = 1 (triplete, 2ħ²).
#  Proyectores:  P_triplete = Ŝ²/2 ,  P_singlete = 1 − Ŝ²/2.
#  El operador de espín no actúa sobre la parte espacial.
# =====================================================================================


def _sumar(destino, base, c):
    destino[base] = sp.simplify(destino.get(base, 0) + c)
    if destino[base] == 0:
        del destino[base]


def combinar(a, b, ca=1, cb=1):
    """ca·a + cb·b para estados (diccionarios)."""
    out = {}
    for base, c in a.items():
        _sumar(out, base, ca * c)
    for base, c in b.items():
        _sumar(out, base, cb * c)
    return out


def escalar(a, k):
    return {base: sp.simplify(k * c) for base, c in a.items() if sp.simplify(k * c) != 0}


def aplicar_sz(estado):
    """Ŝ_z Ψ (en unidades de ħ)."""
    out = {}
    for base, c in estado.items():
        _, s1, _, s2 = base
        m = sp.Rational(1, 2) * ((1 if s1 == "alpha" else -1) + (1 if s2 == "alpha" else -1))
        _sumar(out, base, c * m)
    return out


def aplicar_s2(estado):
    """Ŝ² Ψ (en unidades de ħ²)."""
    out = {}
    for base, c in estado.items():
        o1, s1, o2, s2 = base
        _sumar(out, base, c * (2 if s1 == s2 else 1))
        if s1 != s2:                                   # ŝ₁₊ŝ₂₋ + ŝ₁₋ŝ₂₊ intercambia los espines
            _sumar(out, (o1, s2, o2, s1), c)
    return out


def intercambiar_electrones(estado):
    """Operador de intercambio P₁₂: el electrón 1 pasa a ser el 2 (orbital y espín)."""
    return {(o2, s2, o1, s1): c for (o1, s1, o2, s2), c in estado.items()}


def norma2(estado):
    """⟨Ψ|Ψ⟩ con la base ortonormal (kets distintos son ortogonales)."""
    return sp.simplify(sum(sp.Abs(c) ** 2 for c in estado.values()))


def autovalor(estado, resultado):
    """λ si resultado = λ·estado; None si no son proporcionales."""
    if not estado:
        raise ValueError("El estado no puede ser nulo.")
    if not resultado:
        return sp.Integer(0)
    if set(estado) != set(resultado):
        return None
    lam = None
    for base, c in estado.items():
        cand = sp.simplify(resultado[base] / c)
        if lam is None:
            lam = cand
        elif sp.simplify(cand - lam) != 0:
            return None
    return sp.simplify(lam)


def probabilidades(estado, partes):
    """Probabilidad de cada componente: ‖parte‖² / ‖Ψ‖². Devuelve None si hay coeficientes simbólicos."""
    if any(c.free_symbols for c in estado.values()):
        return None
    total = norma2(estado)
    return [sp.nsimplify(sp.simplify(norma2(p) / total)) for p in partes]


def prob_ms(estado):
    """Probabilidad de medir M_S = +1, 0, −1 (None si hay símbolos)."""
    if any(c.free_symbols for c in estado.values()):
        return None
    total = norma2(estado)
    p = {1: 0, 0: 0, -1: 0}
    for (_, s1, _, s2), c in estado.items():
        m = ((1 if s1 == "alpha" else -1) + (1 if s2 == "alpha" else -1)) // 2
        p[m] += sp.Abs(c) ** 2
    return {m: sp.nsimplify(sp.simplify(v / total)) for m, v in p.items()}


def simetria_intercambio(estado):
    """'antisimétrica', 'simétrica' o 'ninguna' frente al intercambio de los dos electrones."""
    inter = intercambiar_electrones(estado)
    if not combinar(estado, inter, 1, 1):
        return "antisimétrica"
    if not combinar(estado, inter, 1, -1):
        return "simétrica"
    return "ninguna"


# ------------------------- Formato -------------------------
_SIMB = {"alpha": "α", "beta": "β"}


def texto_coef(c):
    """Coeficiente legible: 1/√2, √3/2, −i, ..."""
    c = sp.simplify(c)
    if c.is_real and c != 0 and not c.free_symbols:
        c2 = sp.nsimplify(c ** 2)
        if c2.is_Rational and not c.is_Rational:
            signo = "-" if c < 0 else ""
            p, q = c2.p, c2.q
            raiz = lambda k: "1" if k == 1 else (f"√{k}" if k < 10 ** 6 else f"√({k})")
            if p == 1:
                return f"{signo}1/{raiz(q)}"
            return f"{signo}{raiz(p)}/{raiz(q)}" if q != 1 else f"{signo}{raiz(p)}"
    s = sp.sstr(c).replace("**", "^").replace("sqrt", "√").replace("I", "i").replace("*", "·")
    return s


def texto_ket(base):
    o1, s1, o2, s2 = base
    return f"|{o1},{_SIMB[s1]};{o2},{_SIMB[s2]}⟩"


def texto_estado(estado):
    """Combinación lineal legible, con orden determinista."""
    if not estado:
        return "0"
    out = ""
    for k, (base, c) in enumerate(sorted(estado.items(), key=lambda kv: kv[0])):
        c = sp.simplify(c)
        neg = c.could_extract_minus_sign()
        mag = sp.simplify(-c if neg else c)
        signo = ("-" if neg else "") if k == 0 else (" - " if neg else " + ")
        if mag == 1:
            cuerpo = texto_ket(base)
        else:
            tc = texto_coef(mag)
            cuerpo = f"({tc}){texto_ket(base)}" if re.search(r"[/√+\-·^]", tc) or "i" in tc else f"{tc}{texto_ket(base)}"
        out += signo + cuerpo
    return out.replace("-", "−")


def texto_num(x):
    return str(sp.nsimplify(x)).replace("**", "^")


def analizar(texto):
    """Calcula todo lo pedido y lo devuelve en un diccionario (el informe solo lo imprime)."""
    psi = leer_estado(texto)
    r = {"psi": psi, "norma2": norma2(psi)}
    r["sz_psi"], r["s2_psi"] = aplicar_sz(psi), aplicar_s2(psi)
    r["sz"], r["s2"] = autovalor(psi, r["sz_psi"]), autovalor(psi, r["s2_psi"])
    r["triplete"] = escalar(r["s2_psi"], sp.Rational(1, 2))
    r["singlete"] = combinar(psi, r["triplete"], 1, -1)
    r["p_s2"] = probabilidades(psi, [r["singlete"], r["triplete"]])
    r["p_ms"] = prob_ms(psi)
    r["simetria"] = simetria_intercambio(psi)
    if r["s2"] is not None:
        for s in (0, 1):
            if sp.simplify(r["s2"] - s * (s + 1)) == 0:
                r["s"] = s
    return r


In [4]:
#@title 4) Resultados en texto { display-mode: "form" }
def informe(texto):
    """Imprime el análisis de espín del estado escrito en notación de Dirac."""
    titulo_caja("QUANTUM LAB — ESPÍN TOTAL DE DOS ELECTRONES")
    seccion("ESTADO DE ENTRADA")
    dato("Ψ escrita", texto.strip())
    diag = Diagnostico()
    try:
        r = analizar(texto)
    except (ValueError, SyntaxError, TypeError) as error:
        diag.error(str(error))
        diag.imprimir()
        return None
    psi = r["psi"]
    parrafo("Ψ = " + texto_estado(psi))
    simb = any(c.free_symbols for c in psi.values())
    dato("Número de kets", len(psi))
    dato("⟨Ψ|Ψ⟩", texto_num(r["norma2"]))
    if not simb and sp.simplify(r["norma2"] - 1) != 0:
        diag.nota(f"Ψ no está normalizada (⟨Ψ|Ψ⟩ = {texto_num(r['norma2'])}). Los autovalores no dependen de la "
                  "normalización; las probabilidades se calculan ya normalizadas.")
    if simb:
        diag.nota("Hay coeficientes simbólicos: los autovalores se calculan igualmente, pero no las probabilidades.")
    if r["simetria"] == "antisimétrica":
        diag.nota("Ψ es antisimétrica frente al intercambio de los dos electrones: cumple el principio de Pauli.")
    elif r["simetria"] == "simétrica":
        diag.aviso("Ψ es simétrica frente al intercambio de los electrones: NO cumple el postulado de antisimetría "
                   "(principio de Pauli); no describe un estado físico de dos electrones.")
    else:
        diag.aviso("Ψ no es ni simétrica ni antisimétrica frente al intercambio de los electrones: no cumple el "
                   "principio de Pauli tal como está escrita. Para antisimetrizarla use un determinante de Slater, "
                   "p. ej. Slater(1sα, 2sβ). El valor de S² se calcula igualmente.")
    diag.imprimir("VALIDEZ FÍSICA DE LA FUNCIÓN")

    titulo_caja("COMPONENTE z DEL ESPÍN TOTAL  (Ŝ_z)")
    if r["sz"] is not None:
        dato("¿Autofunción de Ŝ_z?", "sí")
        dato("Autovalor  M_S ħ", f"{texto_num(r['sz'])} ħ")
    else:
        dato("¿Autofunción de Ŝ_z?", "no")
        dato("Ŝ_z Ψ", texto_estado(r["sz_psi"]) + "   (en unidades de ħ)")
        if r["p_ms"] is not None:
            print()
            tabla(["M_S", "Probabilidad"], [(m, texto_num(p)) for m, p in r["p_ms"].items() if p != 0], sangria=3)

    titulo_caja("MOMENTO ANGULAR DE ESPÍN TOTAL  (Ŝ²)")
    if r["s2"] is not None:
        s = r.get("s")
        dato("¿Autofunción de Ŝ²?", "sí")
        dato("Autovalor  S²", f"{texto_num(r['s2'])} ħ²")
        if s is not None:
            dato("Número cuántico de espín total s", s)
            dato("Módulo  |S| = √(s(s+1)) ħ", "0" if s == 0 else "√2 ħ  ≈ 1.4142 ħ")
            dato("Multiplicidad 2s+1", 2 * s + 1)
            dato("Término", "singlete" if s == 0 else "triplete")
    else:
        dato("¿Autofunción de Ŝ²?", "no")
        dato("Ŝ²Ψ (función resultante)", texto_estado(r["s2_psi"]) + "   (en unidades de ħ²)")
        seccion("DESCOMPOSICIÓN EN SINGLETE Y TRIPLETE")
        dato("Componente singlete (S² = 0)", texto_estado(r["singlete"]))
        dato("Componente triplete (S² = 2ħ²)", texto_estado(r["triplete"]))
        if r["p_s2"] is not None:
            print()
            tabla(["Resultado de medir S²", "s", "Probabilidad"],
                  [("0", 0, texto_num(r["p_s2"][0])), ("2 ħ²", 1, texto_num(r["p_s2"][1]))], sangria=3)

    seccion("RESUMEN")
    filas = [("Ŝ_z", f"{texto_num(r['sz'])} ħ" if r["sz"] is not None else "no es autofunción"),
             ("Ŝ²", f"{texto_num(r['s2'])} ħ²" if r["s2"] is not None else "no es autofunción"),
             ("Intercambio de electrones", r["simetria"])]
    tabla(["Operador", "Resultado"], filas, sangria=3, alinear="ll")

    seccion("INTERPRETACIÓN FÍSICA")
    if r["s2"] is not None:
        vinetas([
            "Ψ es autofunción de Ŝ²: una medida del espín total da siempre el mismo valor "
            + ("(singlete: los espines están apareados, S = 0)." if r.get("s") == 0 else "(triplete: espines paralelos en el sentido de S = 1)."),
        ])
    else:
        vinetas(["Ψ es una superposición de singlete y triplete: medir S² da 0 o 2ħ² con las probabilidades indicadas, "
                 "y tras la medida el sistema queda en la componente correspondiente."])
    vinetas(["Ŝ_z y Ŝ² conmutan, así que puede haber estados con ambos valores definidos; ŝ_z no actúa sobre la parte "
             "espacial del espín-orbital."])
    return r


In [5]:
#@title 5) Pruebas (casos conocidos y construcción matricial independiente) { display-mode: "form" }
def ejecutar_pruebas():
    import io, contextlib
    resultados = []

    def prueba(nombre, condicion, detalle=""):
        resultados.append((nombre, bool(condicion), detalle))

    def an(t):
        return analizar(t)

    S = sp.Rational
    # --- A) Verificación independiente con matrices de Pauli (espacio de espín 4×4) ---
    sx, sy, sz_ = (np.array(m, dtype=complex) / 2 for m in ([[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]]))
    I2 = np.eye(2)
    S2_mat = sum(np.kron(s, I2) @ np.kron(s, I2) + np.kron(I2, s) @ np.kron(I2, s) + 2 * np.kron(s, I2) @ np.kron(I2, s)
                 for s in (sx, sy, sz_))
    Sz_mat = np.kron(sz_, I2) + np.kron(I2, sz_)
    prueba("Matriz S² (Pauli): autovalores {0, 2, 2, 2}", np.allclose(sorted(np.linalg.eigvalsh(S2_mat)), [0, 2, 2, 2]))
    orden = [("alpha", "alpha"), ("alpha", "beta"), ("beta", "alpha"), ("beta", "beta")]
    rng = np.random.default_rng(7)
    ok_s2 = ok_sz = True
    for _ in range(25):
        v = rng.normal(size=4) + 1j * rng.normal(size=4)
        est = {("1s", a, "2s", b): sp.Float(v[k].real) + sp.I * sp.Float(v[k].imag) for k, (a, b) in enumerate(orden)}
        for oper, mat in ((aplicar_s2, S2_mat), (aplicar_sz, Sz_mat)):
            res = oper(est)
            w = np.array([complex(res.get(("1s", a, "2s", b), 0)) for a, b in orden])
            if not np.allclose(w, mat @ v, atol=1e-9):
                ok_s2 = ok_s2 and oper is not aplicar_s2
                ok_sz = ok_sz and oper is not aplicar_sz
    prueba("Ŝ² del cuaderno = matriz de Pauli (25 estados aleatorios)", ok_s2)
    prueba("Ŝ_z del cuaderno = matriz de Pauli (25 estados aleatorios)", ok_sz)

    # --- B) Casos conocidos ---
    r = an("(|1s,α;2s,β> - |1s,β;2s,α>)/√2")
    prueba("Singlete de espín: S² = 0, S_z = 0", r["s2"] == 0 and r["sz"] == 0 and r["s"] == 0)
    for nombre, ket, m in (("triplete M=+1", "|1sα;2sα>", 1), ("triplete M=−1", "|1sβ;2sβ>", -1)):
        r = an(ket)
        prueba(f"{nombre}: S² = 2, S_z = {m}", r["s2"] == 2 and r["sz"] == m and r["s"] == 1)
    r = an("(|1sα;2sβ> + |1sβ;2sα>)/sqrt(2)")
    prueba("Triplete M=0: S² = 2, S_z = 0", r["s2"] == 2 and r["sz"] == 0)
    r = an("|1sα;2sβ>")
    prueba("|1sα;2sβ>: S_z = 0, NO autofunción de S²", r["sz"] == 0 and r["s2"] is None)
    prueba("  …y se descompone 50 % singlete / 50 % triplete", r["p_s2"] == [S(1, 2), S(1, 2)], str(r["p_s2"]))
    prueba("  …con S²Ψ = Ψ + |1sβ;2sα>", r["s2_psi"] == {("1s", "alpha", "2s", "beta"): 1, ("1s", "beta", "2s", "alpha"): 1})
    r = an("|1sα;2sα> + |1sβ;2sβ>")
    prueba("αα + ββ: S² = 2 pero NO autofunción de S_z", r["s2"] == 2 and r["sz"] is None)
    prueba("  …M_S = ±1 con probabilidad 1/2", r["p_ms"] == {1: S(1, 2), 0: 0, -1: S(1, 2)}, str(r["p_ms"]))
    r = an("Slater(1sα, 2sβ)")
    prueba("Slater(1sα,2sβ): antisimétrica, S_z = 0, no autofunción de S²",
           r["simetria"] == "antisimétrica" and r["sz"] == 0 and r["s2"] is None)
    prueba("  …50 % singlete / 50 % triplete", r["p_s2"] == [S(1, 2), S(1, 2)])
    r = an("Slater(1sα, 1sβ)")
    prueba("He 1s² = Slater(1sα,1sβ): S² = 0, S_z = 0, antisimétrica", r["s2"] == 0 and r["sz"] == 0 and r["simetria"] == "antisimétrica")
    r = an("(|1sα;2sα> - |2sα;1sα>)/√2")
    prueba("Triplete con parte espacial antisimétrica: S² = 2, S_z = 1, antisimétrica",
           r["s2"] == 2 and r["sz"] == 1 and r["simetria"] == "antisimétrica")
    r = an("(|1sα;2sβ> - |1sβ;2sα> + |2sα;1sβ> - |2sβ;1sα>)/2")
    prueba("Singlete 1s2s totalmente antisimétrico: S² = 0 y antisimétrico", r["s2"] == 0 and r["simetria"] == "antisimétrica")
    r = an("|1sα;1sα>")
    prueba("Dos electrones en 1sα: simétrica (viola Pauli)", r["simetria"] == "simétrica")

    # --- C) Lectura de la entrada ---
    ref = an("(|1sα;2sβ> - |1sβ;2sα>)/sqrt(2)")["psi"]
    for variante in ("1/√2 (|1s,alpha;2s,beta> - |1s,beta;2s,alpha>)", "Ψ = (|1sα;2sβ⟩ - |1sβ;2sα⟩)*2^(-1/2)",
                     "(|1s up;2s down> - |1s down;2s up>)/2^(1/2)", "0.5*sqrt(2)*(|1sα;2sβ> - |1sβ;2sα>)"):
        prueba(f"Variantes de escritura: {variante[:38]}", an(variante)["psi"] == ref)
    r = an("c|1sα;2sβ> + d|1sβ;2sα>")
    prueba("Coeficientes simbólicos c, d", len(r["psi"]) == 2)
    r = an("i|1sα;2sα>")
    prueba("Coeficiente imaginario: autovalores sin cambio", r["s2"] == 2 and r["sz"] == 1)
    prueba("2pz ≡ 2p0 y 2p−1 se acepta", normalizar_orbital("2pz") == "2p0" and normalizar_orbital("2p−1") == "2p-1")
    for malo in ("1p", "2p+2", "3f", "0s", "2q"):
        try:
            normalizar_orbital(malo)
            prueba(f"Rechaza el orbital {malo}", False)
        except ValueError:
            prueba(f"Rechaza el orbital {malo}", True)
    for malo in ("|1sα>", "1sα;2sβ", "|1sγ;2sβ>", "|1sα;2sβ>*|1sα;2sα>"):
        try:
            analizar(malo)
            prueba(f"Rechaza la entrada {malo}", False)
        except (ValueError, SyntaxError, TypeError):
            prueba(f"Rechaza la entrada {malo}", True)
    return informe_pruebas(resultados, "PRUEBAS — ESPÍN DE DOS ELECTRONES")


_pruebas_ok = ejecutar_pruebas()



╔══════════════════════════════════════════════════════════════════════════════╗
║                      PRUEBAS — ESPÍN DE DOS ELECTRONES                       ║
╚══════════════════════════════════════════════════════════════════════════════╝
 ┌─────────────────────────────────────────────────────────────────────────────┬─────────────┬───────────────────────────┐
 │                                    Prueba                                   │  Resultado  │          Detalle          │
 ├─────────────────────────────────────────────────────────────────────────────┼─────────────┼───────────────────────────┤
 │ Matriz S² (Pauli): autovalores {0, 2, 2, 2}                                 │      OK     │                           │
 │ Ŝ² del cuaderno = matriz de Pauli (25 estados aleatorios)                   │      OK     │                           │
 │ Ŝ_z del cuaderno = matriz de Pauli (25 estados aleatorios)                  │      OK     │                           │
 │ Singlete de esp

In [6]:
#@title 6) Entrada del usuario: escribe Ψ en notación de Dirac { display-mode: "form" }
# =====================================================================================
#  Primero el electrón 1 y después el electrón 2, separados por «;». Espín α o β.
#  Ejemplos:   (|1s,α;2s,β⟩ - |1s,β;2s,α⟩)/√2          Slater(1sα, 2sβ)          |1sα;2sα⟩
# =====================================================================================
psi = "(|1s,alpha;2s,beta> - |1s,beta;2s,alpha>)/sqrt(2)"   #@param {type:"string"}

resultado = informe(psi)



╔══════════════════════════════════════════════════════════════════════════════╗
║                 QUANTUM LAB — ESPÍN TOTAL DE DOS ELECTRONES                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

────────────────────────────────────────────────────────────────────────────────
ESTADO DE ENTRADA
────────────────────────────────────────────────────────────────────────────────
   Ψ escrita                               : (|1s,alpha;2s,beta> - |1s,beta;2s,alpha>)/sqrt(2)
   Ψ = (1/√2)|1s,α;2s,β⟩ − (1/√2)|1s,β;2s,α⟩
   Número de kets                          : 2
   ⟨Ψ|Ψ⟩                                   : 1

╔══════════════════════════════════════════════════════════════════════════════╗
║                         VALIDEZ FÍSICA DE LA FUNCIÓN                         ║
╚══════════════════════════════════════════════════════════════════════════════╝
   [AVISO] Ψ no es ni simétrica ni antisimétrica frente al intercambio de los
           electron

In [7]:
#@title 7) Galería de ejemplos (resumen en una tabla) { display-mode: "form" }
EJEMPLOS = [
    ("Singlete 1s2s (espín)", "(|1sα;2sβ> - |1sβ;2sα>)/√2"),
    ("Triplete M=+1", "|1sα;2sα>"),
    ("Triplete M=0", "(|1sα;2sβ> + |1sβ;2sα>)/√2"),
    ("Producto |1sα;2sβ>", "|1sα;2sβ>"),
    ("He 1s² (Slater)", "Slater(1sα, 1sβ)"),
    ("Slater(1sα, 2sβ)", "Slater(1sα, 2sβ)"),
    ("αα + ββ", "|1sα;2sα> + |1sβ;2sβ>"),
    ("Con orbital 2p0", "(|2p0,α;3d+1,β> - |2p0,β;3d+1,α>)/√2"),
]
titulo_caja("GALERÍA DE EJEMPLOS")
filas = []
for nombre, texto in EJEMPLOS:
    r = analizar(texto)
    filas.append((nombre,
                  f"{texto_num(r['s2'])} ħ²" if r["s2"] is not None else "no",
                  f"{texto_num(r['sz'])} ħ" if r["sz"] is not None else "no",
                  r["simetria"],
                  "—" if r["s2"] is not None else f"{texto_num(r['p_s2'][0])} / {texto_num(r['p_s2'][1])}"))
tabla(["Estado", "S²", "S_z", "Intercambio", "P(singlete) / P(triplete)"], filas, sangria=1, alinear="lcccc")



╔══════════════════════════════════════════════════════════════════════════════╗
║                             GALERÍA DE EJEMPLOS                              ║
╚══════════════════════════════════════════════════════════════════════════════╝


 ┌─────────────────────────┬────────┬───────┬─────────────────┬─────────────────────────────┐
 │          Estado         │   S²   │  S_z  │   Intercambio   │  P(singlete) / P(triplete)  │
 ├─────────────────────────┼────────┼───────┼─────────────────┼─────────────────────────────┤
 │ Singlete 1s2s (espín)   │  0 ħ²  │  0 ħ  │     ninguna     │              —              │
 │ Triplete M=+1           │  2 ħ²  │  1 ħ  │     ninguna     │              —              │
 │ Triplete M=0            │  2 ħ²  │  0 ħ  │     ninguna     │              —              │
 │ Producto |1sα;2sβ>      │   no   │  0 ħ  │     ninguna     │          1/2 / 1/2          │
 │ He 1s² (Slater)         │  0 ħ²  │  0 ħ  │  antisimétrica  │              —              │
 │ Slater(1sα, 2sβ)        │   no   │  0 ħ  │  antisimétrica  │          1/2 / 1/2          │
 │ αα + ββ                 │  2 ħ²  │   no  │     ninguna     │              —              │
 │ Con orbital 2p0         │  0 ħ²  │  0 ħ  │     ninguna   